In [ ]:
import sqlite3
import polars as pl

# 1. Připojení k naší SQLite databázi
conn = sqlite3.connect("tests.sqlite")

# 2. SQL dotaz, který spočítá, kolik otázek má a kolik nemá 'explanation'
query_stats = """
SELECT
    CASE
        WHEN explanation IS NULL OR TRIM(explanation) = '' THEN 'Chybí vysvětlení (NULL/Prázdné)'
        ELSE 'Má vysvětlení'
    END AS stav_vysvetleni,
    COUNT(*) AS pocet_otazek,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM questions), 2) AS procento
FROM questions
GROUP BY stav_vysvetleni
"""

# 3. Načtení a zobrazení celkových statistik
df_stats = pl.read_database(query=query_stats, connection=conn)
print("--- CELKOVÁ STATISTIKA VYSVĚTLENÍ ---")
display(df_stats)

print("\n")

# 4. (Volitelné) Ukaž mi ukázku 5 otázek, kterým vysvětlení CHYBÍ
query_missing = """
SELECT id, text AS text_otazky, category
FROM questions
WHERE explanation IS NULL OR TRIM(explanation) = ''
LIMIT 5
"""

df_missing = pl.read_database(query=query_missing, connection=conn)
print("--- UKÁZKA 5 OTÁZEK BEZ VYSVĚTLENÍ ---")
display(df_missing)

# Zavření spojení
conn.close()

In [29]:
import sqlite3
import os
import concurrent.futures
from openai import OpenAI
from tqdm.auto import tqdm

# --- 1. KONFIGURACE e-INFRA API ---
API_KEY = os.getenv("E_INFRA_API_TOKEN", "sk-d8bcb1006a8e47659629e1b4191a2918")
BASE_URL = "https://llm.ai.e-infra.cz/v1"
MODEL_NAME = "deepseek-v3.2-thinking"

# Počet souběžných dotazů (vláken)
MAX_WORKERS = 4

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

def fetch_explanation_from_api(row):
    """Pracovní funkce pro vlákno - pouze stáhne data, nezapisuje do DB!"""
    q_id, question_text, opt_a, opt_b, opt_c, correct_opt = row

    prompt = f"""
    Jsi expertní instruktor letectví a paraglidingu apod. Tvým úkolem je vysvětlit studentovi testovou otázku.

    Znění otázky: {question_text}
    Možnosti:
    A) {opt_a}
    B) {opt_b}
    C) {opt_c}

    Správná odpověď v testu je: {correct_opt}

    Napiš stručné, jasné a odborně přesné vysvětlení v češtině, proč je tato odpověď správná. Dále V případě že to dává smysl napiš i proč ty jiné odpovědi jsou špatně.
    Dodej studentovi kontext, neopakuj jen otrocky znění té správné odpovědi. Neodkazuj se na písmena odpovedi, protože ty se můžoum měnit mezi randomizovanmi běhy otázek.
    DŮLEŽITÉ: Nepoužívej absolutně žádné formátování textu (žádný markdown, hvězdičky pro tučné písmo, kurzívu ani odrážky). Odpovídej pouze v čistém prostém textu.
    """

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Jsi užitečný a expertní AI asistent pro vysvětlování zkušebních testů. Odpovídáš vždy pouze v čistém prostém textu bez jakéhokoliv formátování."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=500 # Lehce zvednuto, DeepSeek thinking modely občas generují delší text kvůli 'přemýšlení'
        )
        explanation = response.choices[0].message.content.strip()

        # Pojistka: manuální odstranění nejčastějších znaků pro Markdown formátování
        explanation = explanation.replace('**', '').replace('*', '').replace('`', '').replace('#', '').strip()

        return (q_id, explanation, None) # Vracíme ID, vysvětlení a žádnou chybu
    except Exception as e:
        return (q_id, None, str(e)) # Vracíme ID, žádné vysvětlení a text chyby

# --- 2. PRÁCE S DATABÁZÍ (Příprava) ---
# Používáme timeout=30 pro jistotu, kdyby byla databáze chvíli zamčená odjinud
conn = sqlite3.connect("tests.sqlite", timeout=30)
cursor = conn.cursor()



# Vytáhneme otázky k vysvětlení (nyní všechny, co mají správnou odpověď)
cursor.execute("""
    SELECT id, text, option_a, option_b, option_c, correct_option
    FROM questions
    WHERE correct_option IS NOT NULL
""")
questions_to_process = cursor.fetchall()

print(f"Nalezeno {len(questions_to_process)} otázek čekajících na vysvětlení.")
print(f"Spouštím masivní paralelní stahování ({MAX_WORKERS} vláken najednou)...")

# --- 3. PARALELNÍ ZPRACOVÁNÍ A BEZPEČNÝ ZÁPIS ---
chyby = 0
uspesne = 0

# Vytvoření bazénu vláken (Thread Pool)
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Namapujeme všechny řádky do vláken
    budouci_vysledky = {executor.submit(fetch_explanation_from_api, row): row for row in questions_to_process}

    # as_completed nám dává výsledky přesně tak, jak chodí z API (kdo dřív přijde, ten dřív mele)
    for future in tqdm(concurrent.futures.as_completed(budouci_vysledky), total=len(questions_to_process), desc="Generování vysvětlení"):
        q_id, explanation, error = future.result()

        if explanation:
            # ZDE BEZPEČNĚ ZAPISUJE POUZE HLAVNÍ VLÁKNO (Žádný Database is locked!)
            cursor.execute(
                "UPDATE questions SET explanation = ? WHERE id = ?",
                (explanation, q_id)
            )
            conn.commit()
            uspesne += 1
        elif error:
            chyby += 1
            # Chyby si nevypisujeme hned, aby to nerozbilo grafiku progress baru,
            # ale můžeš si odkomentovat řádek níže, pokud chceš vidět detaily:
            tqdm.write(f"Chyba u ID {q_id}: {error}")

conn.close()
print("=" * 50)
print(f"Hotovo! Úspěšně obohaceno: {uspesne} otázek. Selhalo API volání: {chyby} otázek.")

Nalezeno 1565 otázek čekajících na vysvětlení.
Spouštím masivní paralelní stahování (4 vláken najednou)...


Generování vysvětlení:   0%|          | 0/1565 [00:01<?, ?it/s]


Chyba u ID ba83b309e2da4b029861129b6677da11: Error code: 429 - {'error': {'message': 'Rate limit exceeded for api_key: 891dcadde961aec4a0b49db2f49f5bbe3a95d33275395e0d6ae05bf96f0d0fd1. Limit type: max_parallel_requests. Current limit: 4, Remaining: 0. Limit resets at: 2026-03-14 20:08:58 UTC', 'type': 'None', 'param': 'None', 'code': '429'}}


KeyboardInterrupt: 

In [6]:
import sqlite3
import os
from openai import OpenAI
from tqdm import tqdm

# --- 1. KONFIGURACE e-INFRA API ---
API_KEY = os.getenv("E_INFRA_API_TOKEN", "sk-eece2562327b4596aafd068eb64f38b2")
BASE_URL = "https://llm.ai.e-infra.cz/v1"
MODEL_NAME = "deepseek-v3.2-thinking"

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

def navrhni_kategorie_pro_davku(chunk_questions):
    """Zpracuje jednu menší dávku otázek a navrhne pro ni kategorie."""
    q_list = "\n".join([f"- {q}" for q in chunk_questions])

    prompt = f"""
    Analyzuj následující dávku testových otázek z teoretických leteckých zkoušek.

    Otázky:
    {q_list}

    Navrhni seznam kategorií, které pokrývají témata POUZE v této konkrétní dávce otázek.
    Vypiš POUZE názvy kategorií jako odrážkový seznam, nic víc.
    """

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Jsi analytik. Odpovídáš pouze stručným seznamem."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=300
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Chyba při zpracování dávky: {e}")
        return ""

def sluc_kategorie(vsechny_navrhy):
    """Vezme dílčí návrhy ze všech dávek a vytvoří jeden sjednocený strom."""
    prompt = f"""
    Zde jsou hrubé návrhy kategorií, které vznikly analýzou několika různých dávek leteckých testových otázek:

    {vsechny_navrhy}

    Tvým úkolem je tyto dílčí seznamy SJEDNOTIT a VYČISTIT.
    1. Odstraň duplicity a sluč velmi podobné kategorie pod jeden výstižný název.
    2. Vytvoř UNIVERZÁLNÍch 8 až 15 hlavních kategorií např. ULL vs. Paragliding).
    3. Cílem je, aby do tohoto finálního seznamu šla jednoznačně zařadit jakákoliv otázka z databáze.

    Vypiš POUZE finální sjednocený odrážkový seznam. Nepiš žádný úvod ani závěr.
    """

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Jsi hlavní examinátor a systémový architekt pro centrální databázi leteckých testů."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2, # Nízká teplota pro precizní deduplikaci
            max_tokens=800
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Chyba při finálním slučování: {e}"

# --- 2. PRÁCE S DATABÁZÍ A ZPRACOVÁNÍ ---
def main():
    conn = sqlite3.connect("tests.sqlite")
    cursor = conn.cursor()

    print("Vytahuji velký reprezentativní vzorek napříč CELOU databází...")

    # Vytáhneme 1000 náhodných otázek (dostatečně velký vzorek pro zachycení všech témat)
    cursor.execute("""
        SELECT text
        FROM questions
        WHERE text IS NOT NULL
        ORDER BY RANDOM()
        LIMIT 1000
    """)

    sample_questions = [row[0] for row in cursor.fetchall()]
    conn.close()

    if not sample_questions:
        print("V databázi nejsou žádné otázky.")
        return

    # Rozdělení na dávky (chunks) po 200 otázkách
    chunk_size = 200
    chunks = [sample_questions[i:i + chunk_size] for i in range(0, len(sample_questions), chunk_size)]

    print(f"Vybráno {len(sample_questions)} otázek. Rozděleno do {len(chunks)} dávek po {chunk_size} otázkách.")
    print("=" * 60)

    dilci_vysledky = []

    # Fáze 1: Analýza jednotlivých dávek
    print("KROK 1: Analýza jednotlivých dávek (Map)...")
    for i, chunk in enumerate(tqdm(chunks, desc="Zpracování dávek")):
        navrh_davky = navrhni_kategorie_pro_davku(chunk)
        if navrh_davky:
            dilci_vysledky.append(f"--- Návrhy z dávky {i+1} ---\n{navrh_davky}\n")

    text_vsech_navrhu = "\n".join(dilci_vysledky)

    # Fáze 2: Sloučení
    print("\nKROK 2: Sjednocování a čištění kategorií (Reduce)...")
    finalni_kategorie = sluc_kategorie(text_vsech_navrhu)

    print("\n" + "=" * 60)
    print("✅ FINÁLNÍ NÁVRH GLOBÁLNÍCH KATEGORIÍ:\n")
    print(finalni_kategorie)
    print("\n" + "=" * 60)
    print("Zkontroluj si tento seznam. Pokud ti vyhovuje, můžeme vytvořit tabulku a začít hromadně klasifikovat!")

if __name__ == "__main__":
    main()

Vytahuji velký reprezentativní vzorek napříč CELOU databází...
Vybráno 1000 otázek. Rozděleno do 5 dávek po 200 otázkách.
KROK 1: Analýza jednotlivých dávek (Map)...


Zpracování dávek: 100%|██████████| 5/5 [00:46<00:00,  9.25s/it]



KROK 2: Sjednocování a čištění kategorií (Reduce)...

✅ FINÁLNÍ NÁVRH GLOBÁLNÍCH KATEGORIÍ:

- Meteorologie
- Letecké předpisy a legislativa
- Aerodynamika a letová mechanika
- Konstrukce a systémy letadel
- Navigace a letové přístroje
- Provozní postupy a bezpečnost (včetně nouzových postupů)
- Zdravotní způsobilost a první pomoc
- Specifické typy letadel (UL, vírníky, kluzáky, PPG, PK, SLZ atd.)
- Letištní provoz a komunikace (včetně návěstění)
- Výcvik, kvalifikace a dokumentace pilotů
- Letové výkony, hmotnost a vyvážení
- Údržba a technická dokumentace

Zkontroluj si tento seznam. Pokud ti vyhovuje, můžeme vytvořit tabulku a začít hromadně klasifikovat!


In [17]:
import sqlite3
import os
import time
import concurrent.futures
from openai import OpenAI
from tqdm import tqdm

# --- 1. KONFIGURACE e-INFRA API ---
API_KEY = os.getenv("E_INFRA_API_TOKEN", "sk-d8bcb1006a8e47659629e1b4191a2918")
BASE_URL = "https://llm.ai.e-infra.cz/v1"

# Pro kategorizaci je lepší rychlý model, který nevypisuje "myšlenkové pochody" (<think>)
MODEL_NAME = "deepseek-v3.2-thinking"

# Sníženo z 32 na 8 kvůli ochraně proti Rate Limitingu (přetížení serveru)
MAX_WORKERS = 4

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

# NÁŠ ZLATÝ SLOUČENÝ SEZNAM KATEGORIÍ
KATEGORIE = [
    "Letecké předpisy a legislativa",
    "Všeobecné znalosti letadel",
    "Letové výkony a plánování",
    "Lidská výkonnost, zdravotní způsobilost a první pomoc",
    "Meteorologie",
    "Navigace a letové přístroje",
    "Provozní postupy a bezpečnost",
    "Principy letu a aerodynamika",
    "Komunikace a letištní provoz",
    "Specifické typy letadel"
]

def setup_database(conn):
    """Upraví schéma databáze pro relační kategorie "po staru" - vytvořením nové tabulky a překopírováním."""
    cursor = conn.cursor()

    # 1. Vytvoření samostatné tabulky pro kategorie
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS categories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE
        )
    """)

    # 2. Naplnění tabulky naší zlatou sadou
    for cat in KATEGORIE:
        cursor.execute("INSERT OR IGNORE INTO categories (name) VALUES (?)", (cat,))

    # Zjištění, zda v tabulce ještě straší starý textový sloupec 'category'
    cursor.execute("PRAGMA table_info(questions)")
    columns = [col[1] for col in cursor.fetchall()]

    if 'category' in columns:
        print("  -> Detekován starý textový sloupec 'category'. Spouštím bezpečnou migraci tabulky...")

        # A. Smazání views, které brání manipulaci s tabulkou
        cursor.execute("DROP VIEW IF EXISTS v_question_usage;")
        cursor.execute("DROP VIEW IF EXISTS v_category_question_stats;")

        # B. Vytvoření nové tabulky bez starého sloupce 'category'
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS questions_new (
                id TEXT PRIMARY KEY,
                text TEXT,
                option_a TEXT,
                option_b TEXT,
                option_c TEXT,
                correct_option TEXT,
                points INTEGER,
                explanation TEXT,
                category_id INTEGER REFERENCES categories(id)
            );
        """)

        # C. Překopírování dat (rovnou s namapováním category_id, pokud stará category existovala)
        cursor.execute("""
            INSERT INTO questions_new (id, text, option_a, option_b, option_c, correct_option, points, explanation, category_id)
            SELECT
                q.id, q.text, q.option_a, q.option_b, q.option_c, q.correct_option, q.points, q.explanation,
                c.id as category_id
            FROM questions q
            LEFT JOIN categories c ON c.name = q.category
        """)

        # D. Smazání staré tabulky a přejmenování nové
        cursor.execute("DROP TABLE questions;")
        cursor.execute("ALTER TABLE questions_new RENAME TO questions;")

        # E. Znovuvytvoření pohledů, tentokrát ale navázaných na tabulku categories
        cursor.execute("""
            CREATE VIEW v_question_usage AS
            SELECT
                q.id as question_id,
                q.text as question_text,
                GROUP_CONCAT(t.test_date) as usage_dates
            FROM questions q
            JOIN test_questions tq ON q.id = tq.question_id
            JOIN tests t ON tq.test_id = t.id
            GROUP BY q.id, q.text;
        """)

        cursor.execute("""
            CREATE VIEW v_category_question_stats AS
            SELECT
                c.name as category_name,
                t.test_type,
                q.id as question_id,
                q.text as question_text,
                COUNT(t.id) as occurrence_count,
                MIN(t.test_date) as first_seen,
                MAX(t.test_date) as last_seen
            FROM questions q
            LEFT JOIN categories c ON q.category_id = c.id
            LEFT JOIN test_questions tq ON q.id = tq.question_id
            LEFT JOIN tests t ON tq.test_id = t.id
            WHERE q.category_id IS NOT NULL
              AND t.test_type IS NOT NULL
            GROUP BY q.category_id, t.test_type, q.id, q.text;
        """)

        print("  -> Migrace úspěšná. Databáze je nyní plně relační.")
    else:
        # Pokud už sloupec 'category' neexistuje, zkusíme alespoň přidat 'category_id', pokud chybí
        try:
            cursor.execute("ALTER TABLE questions ADD COLUMN category_id INTEGER REFERENCES categories(id)")
        except sqlite3.OperationalError:
            pass # Sloupec už tam je

    conn.commit()


def klasifikuj_otazku(row):
    """Vlákno, které vezme jednu otázku a zařadí ji do naší pevné struktury kategorií."""
    q_id, q_text, opt_a, opt_b, opt_c, correct_opt = row

    kategorie_text = "\n".join([f"- {k}" for k in KATEGORIE])

    prompt = f"""
    Zařaď následující leteckou testovou otázku do PRÁVĚ JEDNÉ z těchto oficiálních kategorií:

    {kategorie_text}

    Otázka: {q_text}
    A) {opt_a}
    B) {opt_b}
    C) {opt_c}
    Správná odpověď v testu: {correct_opt}

    Vypiš POUZE přesný název vybrané kategorie z nabídky výše. Nic jiného nepiš.
    """

    # Přidán mechanismus opakování (Retry s Exponential Backoff) proti Rate Limitingu
    max_pokusu = 4
    for pokus in range(max_pokusu):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": "Jsi přesný klasifikační algoritmus leteckých testů. Odpovídáš vždy jen jedním řádkem textu."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=50
            )

            # Bezpečnostní kontrola: pokud API vrátí odpověď, ale její text je None
            raw_content = response.choices[0].message.content
            if raw_content is None:
                raise ValueError("API vrátilo prázdnou odpověď (NoneType). Možná blokace filtru obsahu na straně serveru.")

            vysledek = raw_content.strip()

            for oficialni_kategorie in KATEGORIE:
                klicova_slova = " ".join(oficialni_kategorie.split()[:3]).lower()
                if klicova_slova in vysledek.lower():
                    return (q_id, oficialni_kategorie, None)

            return (q_id, vysledek, None)

        except Exception as e:
            error_msg = str(e)
            # Pokud je to selhání API (přetížení), počkáme a zkusíme to znovu (1s, 2s, 4s...)
            if pokus < max_pokusu - 1:
                time.sleep(2 ** pokus)
            else:
                return (q_id, None, error_msg)

# --- 2. PRÁCE S DATABÁZÍ ---
def main():
    conn = sqlite3.connect("tests.sqlite", timeout=30)

    print("Ověřuji a upravuji strukturu databáze na relační model...")
    setup_database(conn)

    cursor = conn.cursor()

    # Získání mapování (název kategorie -> ID kategorie) do paměti pro rychlé ukládání
    cursor.execute("SELECT name, id FROM categories")
    category_map = dict(cursor.fetchall())

    # Vytáhneme POUZE otázky, které stále nemají přiřazené category_id
    cursor.execute("""
        SELECT id, text, option_a, option_b, option_c, correct_option
        FROM questions
        WHERE category_id IS NULL
    """)
    questions_to_process = cursor.fetchall()

    if not questions_to_process:
        print("Všechny otázky v databázi už jsou relačně zařazeny do nějaké kategorie!")
        conn.close()
        return

    print(f"Nalezeno {len(questions_to_process)} nezařazených otázek.")
    print(f"Spouštím masivní paralelní klasifikaci ({MAX_WORKERS} vláken)...")
    print("=" * 60)

    uspesne = 0
    chyby = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        budouci_vysledky = {executor.submit(klasifikuj_otazku, row): row for row in questions_to_process}

        for future in tqdm(concurrent.futures.as_completed(budouci_vysledky), total=len(questions_to_process), desc="Kategorizace otázek"):
            q_id, urcena_kategorie, error = future.result()

            if urcena_kategorie:
                # Najdeme ID kategorie z našeho relačního slovníku
                cat_id = category_map.get(urcena_kategorie)

                if cat_id:
                    # Zápis do DB - uložíme POUZE relační ID kategorie
                    cursor.execute(
                        "UPDATE questions SET category_id = ? WHERE id = ?",
                        (cat_id, q_id)
                    )
                    conn.commit()
                    uspesne += 1
            elif error:
                chyby += 1
                # Vypíše chybu okamžitě a bezpečně bez rozbití progress baru
                tqdm.write(f"[!] Chyba u otázky ID {q_id}: {error}")

    conn.close()
    print("=" * 60)
    print(f"Hotovo! Nově zařazeno otázek: {uspesne}. Selhání API: {chyby}.")

if __name__ == "__main__":
    main()

Ověřuji a upravuji strukturu databáze na relační model...
Nalezeno 217 nezařazených otázek.
Spouštím masivní paralelní klasifikaci (4 vláken)...


Kategorizace otázek:   6%|▌         | 12/217 [00:11<03:20,  1.02it/s]


KeyboardInterrupt: 